# Resumen de Textos con Técnicas Tradicionales

####  Introducción
El resumen automático de documentos es esencial en ingeniería de software y ciencia de datos para ahorrar tiempo y facilitar la toma de decisiones. Consiste en producir una versión condensada del contenido, conservando las ideas más importantes.

####  Aspectos Básicos
Existen dos paradigmas principales de resumen de texto:

- **Extractivo**: Se extraen frases relevantes del documento original.
- **Abstractive**: Se genera nuevo texto que parafrasea el contenido.

Este apartado se enfoca en técnicas extractivas tradicionales:

- **Modelos de frecuencia de palabras**: Utilizan la frecuencia de palabras (TF-IDF) para identificar oraciones importantes.
- **Métodos basados en grafos (TextRank, LexRank)**: Construyen un grafo de oraciones y aplican un método estilo PageRank.
- **Heurísticas de posición**: Asignan puntuaciones más altas a oraciones en posiciones clave.
- **Modelos basados en LSA (Latent Semantic Analysis)**: Utilizan descomposición SVD para identificar oraciones representativas.

# Métodos extractivos de resumen automático

## 1. TF-IDF (Term Frequency - Inverse Document Frequency)
Este método valora la importancia de las palabras según su frecuencia en una oración y su rareza en el conjunto del texto.  
Las oraciones que contienen términos más relevantes reciben mayor puntuación y son seleccionadas para el resumen.

## 2. TextRank
TextRank es un método basado en grafos inspirado en PageRank.  
Cada oración se representa como un nodo y las conexiones entre nodos dependen de la similitud entre oraciones.  
Las oraciones más “centrales” o importantes dentro del grafo se utilizan para construir el resumen.

## 3. LexRank
LexRank también es un método basado en grafos y muy similar a TextRank.  
La diferencia principal es que normalmente utiliza un umbral de similitud para decidir si dos oraciones deben conectarse.  
Así, selecciona las oraciones más representativas del conjunto.

## 4. Heurísticas de posición
Este enfoque asume que la posición de una oración dentro del texto influye en su importancia.  
Por ejemplo, en muchos documentos las primeras oraciones suelen contener la idea principal.  
Por ello, se asigna una puntuación mayor a las oraciones situadas en posiciones clave.

## 5. LSA (Latent Semantic Analysis)
LSA utiliza una matriz de términos y oraciones y aplica una descomposición SVD para detectar temas latentes en el texto.  
A partir de esa representación semántica, identifica las oraciones que mejor representan el contenido global.  
Es útil para capturar información importante aunque no dependa solo de palabras frecuentes.

In [5]:
pip install numpy scikit-learn networkx

Defaulting to user installation because normal site-packages is not writeable
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
   ---------------------------------------- 0.0/8.0 MB ? eta -:--:--
   ---------------------------------------- 0.1/8.0 MB 3.2 MB/s eta 0:00:03
   --- ------------------------------------ 0.6/8.0 MB 10.1 MB/s eta 0:00:01
   --------- ------------------------------ 1.9/8.0 MB 16.9 MB/s eta 0:00:01
   ------------------- -------------------- 4.0/8.0 MB 25.1 MB/s eta 0:00:01
   -------------------------------------- - 7.7/8.0 MB 37.5 MB/s eta 0:00:01
   ---------------------------------------- 8.0/8.0 MB 36.6 MB/s eta 0:00:00
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
import re
import numpy as np
import networkx as nx
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import TruncatedSVD

#

# =========================
# Texto de entrada
# =========================
text = """
Natural language processing (NLP) is a subfield of linguistics,
computer science, and artificial intelligence concerned with
the interactions between computers and human language.
It is used to apply algorithms to identify and extract the
natural language rules such that the unstructured language data
is converted into a form that computers can understand.
Once the text has been provided, the computer will utilize
algorithms to extract meaning associated with every sentence
and collect the essential data from them.
"""

# =========================
# Utilidades
# =========================
def split_sentences(text):
    text = re.sub(r"\s+", " ", text.strip())
    sentences = re.split(r'(?<=[.!?])\s+', text)
    return [s.strip() for s in sentences if s.strip()]

def get_top_sentences(sentences, scores, n=2):
    ranked_idx = np.argsort(scores)[::-1][:n]      # índices con mayor score
    ranked_idx = sorted(ranked_idx)                # mantener orden original
    return [sentences[i] for i in ranked_idx]

def print_summary(title, summary_sentences):
    print(f"\n--- {title} ---")
    for s in summary_sentences:
        print("-", s)

# =========================
# 1. TF-IDF
# =========================
def summarize_tfidf(sentences, n=2):
    vectorizer = TfidfVectorizer(stop_words="english")
    X = vectorizer.fit_transform(sentences)
    
    # Score de cada oración = suma de los pesos TF-IDF de sus términos
    scores = np.asarray(X.sum(axis=1)).ravel()
    return get_top_sentences(sentences, scores, n)

# =========================
# 2. TextRank
# =========================
def summarize_textrank(sentences, n=2):
    vectorizer = TfidfVectorizer(stop_words="english")
    X = vectorizer.fit_transform(sentences)
    
    sim_matrix = cosine_similarity(X)
    np.fill_diagonal(sim_matrix, 0)

    graph = nx.from_numpy_array(sim_matrix)
    scores_dict = nx.pagerank(graph, weight="weight")
    scores = np.array([scores_dict[i] for i in range(len(sentences))])

    return get_top_sentences(sentences, scores, n)

# =========================
# 3. LexRank
# =========================
def summarize_lexrank(sentences, n=2, threshold=0.1):
    vectorizer = TfidfVectorizer(stop_words="english")
    X = vectorizer.fit_transform(sentences)
    
    sim_matrix = cosine_similarity(X)

    # LexRank clásico suele usar umbral sobre similitud
    adjacency = (sim_matrix >= threshold).astype(float)
    np.fill_diagonal(adjacency, 0)

    graph = nx.from_numpy_array(adjacency)
    scores_dict = nx.pagerank(graph, weight="weight")
    scores = np.array([scores_dict[i] for i in range(len(sentences))])

    return get_top_sentences(sentences, scores, n)

# =========================
# 4. Heurística de posición
# =========================
def summarize_position(sentences, n=2):
    # Más peso a las primeras oraciones
    scores = np.array([1 / (i + 1) for i in range(len(sentences))])
    return get_top_sentences(sentences, scores, n)

# =========================
# 5. LSA (SVD)
# =========================
def summarize_lsa(sentences, n=2):
    vectorizer = TfidfVectorizer(stop_words="english")
    X = vectorizer.fit_transform(sentences)

    # Elegimos un número válido de componentes
    n_components = min(2, X.shape[0], X.shape[1])
    if n_components < 1:
        return sentences[:n]

    svd = TruncatedSVD(n_components=n_components, random_state=42)
    X_lsa = svd.fit_transform(X)

    # Score de cada oración = norma en el espacio latente
    scores = np.linalg.norm(X_lsa, axis=1)
    return get_top_sentences(sentences, scores, n)

# =========================
# Ejecutar todo
# =========================
sentences = split_sentences(text)

print("ORACIONES ORIGINALES:")
for i, s in enumerate(sentences, 1):
    print(f"{i}. {s}")

summary_tfidf = summarize_tfidf(sentences, n=2)
summary_textrank = summarize_textrank(sentences, n=2)
summary_lexrank = summarize_lexrank(sentences, n=2)
summary_position = summarize_position(sentences, n=2)
summary_lsa = summarize_lsa(sentences, n=2)

print_summary("Resumen TF-IDF", summary_tfidf)
print_summary("Resumen TextRank", summary_textrank)
print_summary("Resumen LexRank", summary_lexrank)
print_summary("Resumen Heurística de Posición", summary_position)
print_summary("Resumen LSA (SVD)", summary_lsa)

ORACIONES ORIGINALES:
1. Natural language processing (NLP) is a subfield of linguistics, computer science, and artificial intelligence concerned with the interactions between computers and human language.
2. It is used to apply algorithms to identify and extract the natural language rules such that the unstructured language data is converted into a form that computers can understand.
3. Once the text has been provided, the computer will utilize algorithms to extract meaning associated with every sentence and collect the essential data from them.

--- Resumen TF-IDF ---
- Natural language processing (NLP) is a subfield of linguistics, computer science, and artificial intelligence concerned with the interactions between computers and human language.
- It is used to apply algorithms to identify and extract the natural language rules such that the unstructured language data is converted into a form that computers can understand.

--- Resumen TextRank ---
- Natural language processing (NLP)




# Ejercicio Practico: análisis temático y resumen extractivo multidocumento

## Objetivo
Implementar un sistema capaz de procesar entre **3 y 5 textos** sobre un mismo tema, identificar los **temas principales** del conjunto mediante **LDA (Latent Dirichlet Allocation)** y generar un **resumen extractivo final** utilizando **uno** de los siguientes métodos:

- TF-IDF
- TextRank
- LexRank
- Heurísticas de posición
- LSA

## Descripción del problema
Se dispone de un pequeño conjunto de documentos relacionados entre sí, por ejemplo varias noticias, artículos o textos de divulgación sobre un mismo asunto. El objetivo es construir un sistema que sea capaz de:

1. **analizar el contenido global** de los textos,
2. **detectar los temas principales** presentes en el conjunto documental,
3. **extraer las oraciones más relevantes** para construir un resumen final.

De este modo, el alumno trabajará tanto la parte de **modelado de temas** como la de **resumen automático extractivo**.

## Entrada

El sistema recibirá una lista de entre **3 y 5 textos**, por ejemplo:


docs = [texto1, texto2, texto3]


## Salida Esperada
Temas detectados con LDA:
* Tema 1: language, computer, processing, text, algorithms
* Tema 2: sentence, extract, meaning, data, understand

Método de resumen: TextRank

Resumen final:
1. ...
2. ...
3. ...
4. ...